# CSC2042S Machine Learning @ UCT
# Neural Networks

**Author: Buqwana Xolisile**

This notebook implements a feedforward neural network for image classification on the `Fashion-MNIST` dataset. It begins by defining a custom **Loader** class to read the dataset from CSV files, preprocess the data, and normalize pixel values. The dataset is then divided into training, validation, and test subsets using an 80/20 split of the original training data.

Using PyTorch, the model is built with multiple fully connected layers and nonlinear activation functions such as ReLU. The implementation explores multiclass classification through the softmax output layer, optimized via the cross-entropy loss function and trained using stochastic gradient descent (SGD).

Throughout the notebook, various hyperparameters—including learning rate, number of hidden units, and batch size are tuned to assess their effect on model convergence and generalization. Training progress is monitored via accuracy metrics computed on both training and validation sets.

Finally, model performance is evaluated using the confusion matrix and accuracy metrics on the unseen test data, highlighting the model’s ability to distinguish between similar clothing categories. Visualization of loss curves and prediction results provides insights into training stability and class confusion patterns.

**Outline:**

* [1. Data processing](#1.-data-processing)
* [2. Building and training a Baseline model](#section2)
* [3. Hyperparameter Optimisation Experiment](#section3)
* [4. Analysis](#section4) 

## Imports, Installations, and Downloads

This notebook will make extensive use of some standard Python libraries for scientific computing, namely:
* ``torch`` for performing tensor operations and training neural networks.
* ``torch.nn`` for building model layers such as ``nn.Linear``.
* ``torch.nn.functional`` (``F``) for functions like ``relu`` and ``cross_entropy`` used during training.
* ``torch.utils.data`` for managing datasets using ``Dataset``, ``DataLoader``, and ``random_split``.
* ``torchvision.transforms`` for preprocessing and augmenting image data.
* ``numpy`` for efficient numerical and array computations.
* ``matplotlib.pyplot`` for plotting and visualizing data or results.
* ``sklearn.metrics`` for evaluating performance using ``confusion_matrix``.
* ``pandas`` for working with tabular data structures like DataFrames.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

## Seed
* Set random seed for reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# 1. Data Processing

We will now load the **Fashion-MNIST** dataset, which is a collection of 70,000 grayscale images of 28x28 pixels representing 10 different fashion categories, such as T-shirts, shoes, and bags. The dataset is already split into **60,000 training images** and **10,000 test images** available at: [Fashion-MNIST Kaggle](https://www.kaggle.com/datasets/zalando-research/fashionmnist).

We use the defined class below **Loader** which has the following functions that work as follows:

The **`__init__`** function loads the dataset from a given CSV file using NumPy. It separates the first column as labels and the remaining columns as image pixel values. An optional **transform** can be applied to each image for preprocessing or augmentation.

The **`__len__`** function returns the total number of samples in the dataset. For Fashion-MNIST, this will be 60,000 for training and 10,000 for testing.

The **`__getitem__`** function retrieves a single sample by index. It reshapes the 1D pixel array into a 28x28 image, normalizes the pixel values to the range [0, 1], converts the image and label into PyTorch tensors, and optionally applies any additional transformations provided.

In [ ]:
class Loader():
    """ Fashion-MNIST Loader from CSV files using numpy. """
    def __init__(self, csv_file, transform=None):
        data = np.genfromtxt(csv_file, delimiter=',', skip_header=1)
        self.labels = data[:, 0].astype(np.int64)
        self.pixels = data[:, 1:].astype(np.float32)
        self.transform = transform
        
    def __len__(self):
        return len(self.labels) # Returns the total number of samples in the Fashion-MNIST dataset.
    
    def __getitem__(self, index):
        image = self.pixels[index].reshape(28, 28) # Get the image pixels and reshape to 28x28.
        label = self.labels[index]
        
        image = image / 255.0  # Normalize pixel values to [0, 1].
        
        image = torch.from_numpy(image).unsqueeze(0)  # Adds channel dimension (1, 28, 28).
        label = torch.tensor(label, dtype=torch.long)
        
        if self.transform:  
            image = self.transform(image)
        return image, label

### Load full MNIST dataset
* Using this class, we can create dataset instances and further split the training set into **training** and **validation subsets**. For example, an 80/20 split on the 60,000 training images results in **48,000 training samples** and **12,000 validation samples**.
  

In [ ]:
if __name__ == "__main__":
    data_path = r"kaggle"
    
    train_csv = f"{data_path}\\fashion-mnist_train.csv"
    test_csv = f"{data_path}\\fashion-mnist_test.csv"

    print("Loading...")

    train_dataset = Loader(csv_file=train_csv)
    test_dataset = Loader(csv_file=test_csv)

    print(f"Training samples: {train_dataset.__len__()}")
    print(f"Test samples: {test_dataset.__len__()}")

   
    train_size = int(0.8 * train_dataset.__len__())  # Split training data into train and validation sets 80/20 split.
    val_size = train_dataset.__len__() - train_size

    train_subset, val_subset = random_split(
        train_dataset,
        [train_size, val_size],
    )

    print(f"Training subset: {len(train_subset)}")
    print(f"Validation subset: {len(val_subset)}")

# 2. Building and training a Baseline model

### Feedforward Neural Network

We further define a **`FeedforwardNeuralNetModel`** class, which is a simple artificial neural network which is a multilayer perceptron. The class inherits from `nn.Module`. Inside the class, we define three linear layers and two ReLU activations.

The network architecture consists of as defined by the constructor of the class:

- **Input Layer:** The `nn.Flatten` layer reshapes each input image from a 28×28 matrix into a 784-dimensional vector so it can be fed into the connected layers.

- **Hidden Layer 1:** `nn.Linear(28*28, 128)` connects the 784 input features to 128 neurons, followed by `nn.ReLU()` which applies a non-linear activation.

- **Hidden Layer 2:** `nn.Linear(128, 64)` reduces the 128 features to 64 neurons, again followed by a `nn.ReLU()` activation.

- **Output Layer:** `nn.Linear(64, 10)` maps the 64 features to 10 output neurons, corresponding to the 10 Fashion-MNIST classes.

The **forward** method defined execute computations applied to an input image `x`.

1. The image `x` is first **flattened** into a 1D vector using `self.flatten(x)`.
2. The flattened vector passes through three fully connected layers (`fc1`, `fc2`, `fc3`), with **ReLU activations** applied after the first two layers.
3. The final layer (`fc3`) outputs **raw class scores (logits)** for each of the 10 categories.

Mathematically, this can be represented as:
$$
h_{\mathbf{W}, \mathbf{b}}(\mathbf{x}) = \mathbf{W}^{(3)} \, g^{(2)} \Big( \mathbf{W}^{(2)} \, g^{(1)}(\mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)}) + \mathbf{b}^{(2)} \Big) + \mathbf{b}^{(3)}
$$

$$
\text{final\_output} =
\mathbf{W}^{(3)} \cdot \text{ReLU}^{(2)} \Big( 
\mathbf{W}^{(2)} \cdot \text{ReLU}^{(1)}(\mathbf{W}^{(1)} \cdot \mathbf{x} + \mathbf{b}^{(1)}) + \mathbf{b}^{(2)} 
\Big) + \mathbf{b}^{(3)}
$$






In [ ]:
class FeedforwardNeuralNetModel(nn.Module): 
    def __init__(self):
        super(FeedforwardNeuralNetModel, self).__init__()
        
        self.flatten = nn.Flatten() # Input Layer (flattening layer)

        # Hidden Layer 1 : 128 neurons
        self.fc1 = nn.Linear(28*28, 128) # Linear function 1: 128 ---> 64
        self.relu1 = nn.ReLU() # Non-linearity 1: ReLU activation
        
        # Hidden Layer 2: 64 neurons
        self.fc2 = nn.Linear(128, 64) # Linear function 2: 64 ---> 10
        self.relu2 = nn.ReLU() # Non-linearity 2: ReLU activation
        
        # Output Layer: 10 neurons
        self.fc3 = nn.Linear(64, 10) # Linear function 3 (readout): 64 ---> 10
    
    def forward(self, x):
        final_output = self.fc3(self.relu2(self.fc2(self.relu1(self.fc1(self.flatten(x) ))))) 
        return final_output 

### Mini-Batch Loading
- This block creates **DataLoader** objects to load the training, validation, and test data in **mini-batches**, allowing the model to process smaller chunks of data at a time during training and evaluation.

In [ ]:
if __name__ == "__main__":
    batch_size = 64

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )
    
    print(f"Batch size: {batch_size}")
    print(f"Number of training batches: {len(train_loader)}")
    print(f"Number of validation batches: {len(val_loader)}")
    print(f"Number of test batches: {len(test_loader)}")

### Training

Now we want to train our models and observe how they learn from the input data. To do this, we define the `Trainer` class, which is responsible for handling the training and validation of a neural network model such as `FeedforwardNeuralNetModel`. This class uses the forward pass method defined earlier, loss function, backpropagation, and performance evaluation for multiple epochs.

where the loss function is mathmatically defined as:
 $$
L_{\text{CE}}(\hat{y}, y) = - \sum_{i=1}^{C} y_i \, \log \big( \text{softmax}(z)_i \big)
$$

which uses softmax function defined as:
$$
\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}
$$

finally intergrated together we have:
$$
L_{CE}(\hat{y}, y) = - \sum_{i=1}^{C} y_i \log \left( \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}} \right)
$$

then we have a optimizer defined as:
$$
\hat{\theta} = \arg \min_{\theta} \frac{1}{m} \sum_{i=1}^{m} L_{CE}\big(f(\mathbf{x}^{(i)}; \theta), \mathbf{y}^{(i)}\big)
$$

Where:
$$
{m} = \text{number of batch samples}
$$
$$
\theta = \{ \mathbf{W}^{(1)}, \mathbf{b}^{(1)}, \mathbf{W}^{(2)}, \mathbf{b}^{(2)}, \mathbf{W}^{(3)}, \mathbf{b}^{(3)} \}
$$

finally the optimizer is:
$$
\hat{\theta} = \arg\min_{\theta} \frac{1}{m} \sum_{i=1}^{m} \Bigg[ - \sum_{c=1}^{C} y_c^{(i)} \log \big( \text{softmax}(f(\mathbf{x}^{(i)}; \theta))_c \big) \Bigg]
$$

to further optimize the model we use the gradeint descent defined as:
$$
\theta_{t+1} = \theta_t - \eta \, \nabla_\theta L(f(\mathbf{x}; \theta), \mathbf{y})
$$
$$
\theta_{t+1} = \theta_t - \eta \, \nabla_\theta 
L_{\text{CE}}\Big(\text{softmax}\big(\mathbf{W}^{(3)} \cdot 
\text{ReLU}^{(2)}(\mathbf{W}^{(2)} \cdot 
\text{ReLU}^{(1)}(\mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)}) + \mathbf{b}^{(2)}) + \mathbf{b}^{(3)}\big), \mathbf{y} \Big)
$$

Now for backpropagation the mathmatical notation is defined as follows:

**Gradients for Output Layer (Layer 3):**

$$
\frac{\partial L_{\text{CE}}}{\partial \mathbf{W}^{(3)}} = \big(\text{softmax}(\mathbf{z}) - \mathbf{y}\big) \cdot \mathbf{a}^{(2)T}
$$
Where the activation from the second hidden layer is:

$$
\mathbf{a}^{(2)} = \text{ReLU}(\mathbf{W}^{(2)} \mathbf{a}^{(1)} + \mathbf{b}^{(2)})
$$

Here, $\mathbf{a}^{(2)}$ is the output from hidden layer 2.

**Gradient for Hidden Layer 2 (Layer 2):**

$$
\frac{\partial L}{\partial \mathbf{W}^{(2)}} = 
\frac{\partial L}{\partial \mathbf{a}^{(2)}} \cdot 
\text{ReLU}'(\mathbf{z}^{(2)}) \cdot 
\mathbf{a}^{(1)T}
$$

Where the derivative of ReLU is:

$$
\text{ReLU}'(z) =
\begin{cases} 
1 & \text{if } z > 0 \\
0 & \text{if } z \leq 0
\end{cases}
$$


Using the Chain Rule for backpropagation:

$$
\frac{\partial L}{\partial \mathbf{W}_i} = 
\frac{\partial L}{\partial \mathbf{y}} \cdot 
\frac{\partial \mathbf{y}}{\partial \mathbf{z}} \cdot 
\frac{\partial \mathbf{z}}{\partial \mathbf{W}_i}
$$





The **constructor (`__init__`)** initializes the class with the model to be trained, the training and validation `DataLoader` objects, the loss function (`criterion`), and the optimizer. It also initializes lists to keep track of training losses, validation losses, and validation accuracies across epochs.

The **train** method performs the training process over a specified number of epochs. Each epoch consists of two phases:

1. **Training Phase:**  
   - The model is set to training mode. 
   - For each mini-batch from the training loader, the method performs a forward pass to compute predictions, calculates the loss using the specified criterion, and performs backpropagation to update the model weights using the optimizer.  
   - The average training loss across all batches is computed and recorded.

2. **Validation Phase:**  
   - The model is switched to evaluation mode, disabling gradient computations.  
   - For each mini-batch from the validation loader, the method computes predictions and calculates the loss.  
   - It also determines the number of correct predictions to compute the validation accuracy for the epoch.  
   - The average validation loss and accuracy are recorded.

After each epoch, the training and validation losses, as well as the validation accuracy, are printed.

In [ ]:
class Trainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
    
    def train(self, epochs=10):
        for epoch in range(epochs):
            self.model.train()
            running_loss = 0.0
            correct_train = 0
            total_train = 0
            
            for images, labels in self.train_loader:
                outputs = self.model(images)  # Forward pass
                loss = self.criterion(outputs, labels) # Compute loss function 
                
                self.optimizer.zero_grad()  # Backpropagation
                loss.backward() # Computes ∇L which is gradient of loss with respect to parameters using chain rule.
                
                self.optimizer.step() # Performs θ^(t+1) = θ^t - η∇L
                
                running_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total_train += labels.size(0)
                correct_train += (predicted == labels).sum().item()
            
            train_loss = running_loss / len(self.train_loader)
            train_accuracy = 100 * correct_train / total_train
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_accuracy)
            
            self.model.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            
            with torch.no_grad():
                for images, labels in self.val_loader:
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                    val_loss += loss.item()
                    
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
            
            val_loss = val_loss / len(self.val_loader)
            self.val_losses.append(val_loss)
            
            val_accuracy = 100 * correct / total
            self.val_accuracies.append(val_accuracy)
            
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')
        
        return self.train_losses, self.train_accuracies, self.val_losses, self.val_accuracies

# Training the Model
We now create an instance of the `FeedforwardNeuralNetModel`, define the loss function (`nn.CrossEntropyLoss`) and optimizer (`Adam`), train the model for 10 epochs using the `Trainer` class, and finally print the model's validation accuracy after training.


In [ ]:
if __name__ == "__main__":
    model = FeedforwardNeuralNetModel()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print("Training Baseline Model...")
    trainer = Trainer(model, train_loader, val_loader, criterion, optimizer)
    baseline_train_losses, baseline_train_accuracies, baseline_val_losses, baseline_val_accuracies = trainer.train(epochs=10)

    baseline_val_accuracy = baseline_val_accuracies[-1]
    baseline_train_accuracy = baseline_train_accuracies[-1]
    print(f"\nBaseline Model Final Training Accuracy: {baseline_train_accuracy:.2f}%")
    print(f"Baseline Model Final Validation Accuracy: {baseline_val_accuracy:.2f}%")

    epochs = range(1, len(baseline_train_losses) + 1)
    # Plots   
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, baseline_train_losses, label='Train Loss', marker='o', linewidth=2)
    plt.plot(epochs, baseline_val_losses, label='Validation Loss', marker='s', linewidth=2)
    plt.xlabel('Epoch', fontsize=11)
    plt.ylabel('Loss', fontsize=11)
    plt.title('Baseline Model: Training and Validation Loss', fontsize=12, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, baseline_train_accuracies, label='Train Accuracy', marker='o', linewidth=2)
    plt.plot(epochs, baseline_val_accuracies, label='Validation Accuracy', marker='s', linewidth=2)
    plt.xlabel('Epoch', fontsize=11)
    plt.ylabel('Accuracy (%)', fontsize=11)
    plt.title('Baseline Model: Training and Validation Accuracy', fontsize=12, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 3. Hyperparameter Optimisation Experiment

### Modified Feedforward Neural Network

We now define the `ModifiedFeedforwardNeuralNetModel` class, which extends the previously defined neural network.  
This modified model introduces **flexible hidden layer sizes** and **optional activation functions** (`ReLU`, `Sigmoid`, or `Tanh`) to allow experimentation with different nonlinearities.

The network architecture consists of:

- **Input Layer**: The `nn.Flatten` layer reshapes each input image from a 28×28 matrix into a 784-dimensional vector, preparing it for the fully connected layers.
- **Hidden Layer 1**: `nn.Linear(28*28, hidden1_size)` maps the flattened input to `hidden1_size` neurons (default 128), followed by the chosen activation function.
- **Hidden Layer 2**: `nn.Linear(hidden1_size, hidden2_size)` maps the first hidden layer to `hidden2_size` neurons (default 64), again followed by the same activation.
- **Output Layer**: `nn.Linear(hidden2_size, 10)` maps the features from the second hidden layer to 10 output neurons, one for each Fashion-MNIST class.

The **forward** method defines the sequence of computations performed on an input image `x`:

1. The image `x` is **flattened** into a 1D vector using `self.flatten(x)`.  
2. The flattened input passes through the first fully connected layer `fc1`, followed by the **activation function**.  
3. The output then passes through the second fully connected layer `fc2`, again followed by the **activation**.  
4. Finally, the last fully connected layer `fc3` produces the **raw output scores (logits)** for each class.

This process can be expressed mathematically as:

$$
\text{final\_output}_{\mathbf{W}, \mathbf{b}}(\mathbf{x}) =
\mathbf{W}^{(3)} \cdot \text{Activation}^{(2)} \Big(
\mathbf{W}^{(2)} \cdot \text{Activation}^{(1)}(\mathbf{W}^{(1)} \cdot \mathbf{x} + \mathbf{b}^{(1)})
+ \mathbf{b}^{(2)} \Big) + \mathbf{b}^{(3)}
$$

where:
- $\mathbf{W}^{(i)}$ and $\mathbf{b}^{(i)}$ are the weights and biases of layer *i*,  
- $\text{Activation}^{(i)}$ is the chosen activation function applied after layer *i*,  
- $\text{final\_output}$ represents the logits for all 10 output classes.


In [ ]:
class ModifiedFeedforwardNeuralNetModel(FeedforwardNeuralNetModel):
    def __init__(self, hidden1_size=128, hidden2_size=64, activation='relu'):
        super(ModifiedFeedforwardNeuralNetModel, self).__init__()
        
        self.flatten = nn.Flatten() # Input Layer (flattening layer)
        
        # Hidden Layer 1 
        self.fc1 = nn.Linear(28*28, hidden1_size) # Linear function 1: Input ---> Hidden1
        
        # Hidden Layer 2 
        self.fc2 = nn.Linear(hidden1_size, hidden2_size) # Linear function 2: Hidden1 ---> Hidden2
        
        # Output Layer: 10 neurons
        self.fc3 = nn.Linear(hidden2_size, 10) # Linear function 3 (readout): Hidden2 ---> Output
        
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'sigmoid':
            self.activation = nn.Sigmoid()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        else:
            self.activation = nn.ReLU()
    
    def forward(self, x):
        final_output = self.fc3(self.activation(self.fc2(self.activation(self.fc1(self.flatten(x)))))) 
        return final_output

### **Hyperparameter Optimization**

We will now define a `HyperparameterOptimizer` class to systematically search for the best combination of **learning rate**, **activation function**, and **hidden layer size** for our neural network. This process is called **hyperparameter tuning** and helps improve model performance beyond the baseline.

The class takes in the **training** and **validation DataLoaders** and defines possible values for:

- **Learning rates**: `[0.1, 0.01, 0.001]`
- **Hidden layer sizes (first layer)**: `[64, 128, 256]` 
- **Activation functions**: `['relu', 'sigmoid', 'tanh']`
1. **ReLU (Rectified Linear Unit)**  
   $$
   \text{ReLU}(z) = \max(0, z)
   $$

2. **Sigmoid**  
   $$
   \sigma(z) = \frac{1}{1 + e^{-z}}
   $$

3. **Tanh (Hyperbolic Tangent)**  
   $$
   \tanh(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}
   $$ 

The **`run_optimization`** method performs a full grid search over all combinations of these hyperparameters:

1. For each combination of learning rate, activation function, and hidden layer size:  
   - A new `ModifiedFeedforwardNeuralNetModel` is created with the specified hidden size and activation function.  
   - A loss function `nn.CrossEntropyLoss()` is defined.  
   - The **Mini-batch SGD optimizer** is applied with the chosen learning rate.  
   - The model is trained using the `Trainer` class for the specified number of epochs.  
   - The **final validation accuracy** is recorded along with training and validation losses and accuracy per epoch.

2. After testing all combinations, the method identifies the **best model** based on validation accuracy.


In [ ]:
class HyperparameterOptimizer:
    def __init__(self, train_loader, val_loader):
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.learning_rates = [0.1, 0.01, 0.001]
        self.activations = ['relu', 'sigmoid', 'tanh']  
        self.hidden1_sizes = [64, 128, 256]
        self.results = []
    
    def run_optimization(self, baseline_accuracy, epochs=10):
        total_combinations = len(self.learning_rates) * len(self.activations) * len(self.hidden1_sizes)
        current = 0
        
        for lr in self.learning_rates:
            for activation in self.activations:
                for hidden1 in self.hidden1_sizes:
                    current += 1
                    print(f"\n[{current}/{total_combinations}] Testing: LR={lr}, Activation={activation}, Hidden1={hidden1}")
                    
                    model = ModifiedFeedforwardNeuralNetModel(
                        hidden1_size=hidden1, 
                        hidden2_size=64, 
                        activation=activation
                    )
                    
                    criterion = nn.CrossEntropyLoss()
                    optimizer = torch.optim.SGD(model.parameters(), lr=lr)  # Mini-batch SGD
                    
                    trainer = Trainer(model, self.train_loader, self.val_loader, criterion, optimizer)
                    train_losses, train_accuracies, val_losses, val_accuracies = trainer.train(epochs=epochs)
                    
                    final_val_accuracy = val_accuracies[-1]
                    final_train_accuracy = train_accuracies[-1]
                    self.results.append({
                        'learning_rate': lr,
                        'activation': activation,
                        'hidden1_size': hidden1,
                        'final_val_accuracy': final_val_accuracy,
                        'final_train_accuracy': final_train_accuracy,
                        'train_losses': train_losses,
                        'train_accuracies': train_accuracies,
                        'val_losses': val_losses,
                        'val_accuracies': val_accuracies
                    })
                    
                    print(f"Final Validation Accuracy: {final_val_accuracy:.2f}%")
        
        best_result = max(self.results, key=lambda x: x['final_val_accuracy'])
        
        print(f"\nBest Model:")
        print(f"  Learning Rate: {best_result['learning_rate']}")
        print(f"  Activation: {best_result['activation']}")
        print(f"  Hidden1 Size: {best_result['hidden1_size']}")
        print(f"  Validation Accuracy: {best_result['final_val_accuracy']:.2f}%")
        print(f"  Training Accuracy: {best_result['final_train_accuracy']:.2f}%")
        print(f"  Improvement over Baseline Validation: {best_result['final_val_accuracy'] - baseline_val_accuracy:.2f}%")
        print(f"  Improvement over Baseline Training: {best_result['final_train_accuracy'] - baseline_train_accuracy:.2f}%")

        best_model = ModifiedFeedforwardNeuralNetModel(
            hidden1_size=best_result['hidden1_size'],
            hidden2_size=64,
            activation=best_result['activation']
            )
        
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(best_model.parameters(), lr=best_result['learning_rate'])
        trainer = Trainer(best_model, self.train_loader, self.val_loader, criterion, optimizer)
        trainer.train(epochs=epochs)
        
        return self.results, best_result, best_model

In [ ]:
if __name__ == "__main__":
    hp_optimizer = HyperparameterOptimizer(train_loader, val_loader)
    results, best_result, best_result_model = hp_optimizer.run_optimization(baseline_accuracy, epochs=10)

# 4. Analysis 
### Evaluation of Model Performance

We now define an **`Evaluate`** class to analyze and visualize the outcomes of the hyperparameter optimization experiments.  
This class compares the **best-performing model** from the optimization process with the **baseline model** to assess performance improvements.

The class is initialized with the following parameters:

- **`results`**: A list of dictionaries, each storing a hyperparameter combination and its corresponding validation accuracy, training losses, and validation losses.
- **`baseline_accuracy`**: The validation accuracy of the baseline model (before optimization).
- **`baseline_train_losses`** and **`baseline_val_losses`**: Lists containing training and validation losses of the baseline model for comparison.
- **`best_result`**: configuration with the highest validation accuracy from all tested combinations
- **`class_names`**: A list of Fashion-MNIST class labels

The class provides the following methods:

### Display Results Table

**`display_results_table()`** prints a formatted table of all tested hyperparameter combinations, showing:

- **Learning Rate**
- **Activation Function**
- **Hidden Layer Size**
- **Validation Accuracy (%)**
- **Baseline Accuracy**
- **Best Model Accuracy**
- **Improvement Over Baseline**

### Hyperparameter Impact Visualization
**`plot_hyperparameter_impact()`** generates a bar chart comparing validation accuracies for all hyperparameter combinations, highlighting the best result and showing the baseline as a reference.

### Loss Curve Visualization
**`plot_loss_curves()`** plots:

- Training and validation loss curves for the baseline and best model.

### Confusion Matrix
**`plot_confusion_matrix(predictions, labels)`** generates a heatmap of the confusion matrix to visualize classification errors and class-wise performance.


### Compare Best vs. Baseline

**`analyze_best_vs_baseline()`** prints a summary comparing the best hyperparameter combination against the baseline model and how much tuning the hyperparameters improved the model's performance.

### Hyperparameter Influence 

**`analyze_hyperparameter_influence()`** evaluates the impact of each hyperparameter (learning rate, Activation function and hidden layer) on model performance.  

### Overfitting Analysis

**`analyze_overfitting()`** compares the **training and validation losses** of the baseline model versus the best model to evaluate overfitting. 

### Error Analysis (Confusion Matrix)

**`analyze_errors(confusion_matrix)`** inspects the model’s misclassifications using the **confusion matrix**:

In [ ]:
class Evaluate:
    def __init__(self, results, baseline_accuracy, baseline_train_losses, baseline_val_losses):
        self.results = results
        self.baseline_accuracy = baseline_accuracy
        self.baseline_train_losses = baseline_train_losses
        self.baseline_val_losses = baseline_val_losses
        self.baseline_train_accuracies = baseline_train_accuracies
        self.baseline_val_accuracies = baseline_val_accuracies
        self.best_result = max(results, key=lambda x: x['final_val_accuracy'])
        self.class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

    def display_results_table(self):
        df = pd.DataFrame(self.results)[['learning_rate', 'activation', 'hidden1_size', 'final_val_accuracy']]
        df.columns = ['Learning Rate', 'Activation', 'Hidden1 Size', 'Val Accuracy (%)']
        print("\n" + "="*90)
        print("EXPERIMENTAL RESULTS TABLE")
        print("="*90)
        print(df.to_string(index=False))
        print("="*90)
        print(f"\nBaseline Accuracy: {self.baseline_accuracy:.2f}%")
        print(f"Best Accuracy: {self.best_result['final_val_accuracy']:.2f}%")
        print(f"Improvement: {self.best_result['final_val_accuracy'] - self.baseline_accuracy:.2f}%")
        print("="*90 + "\n")
        return df

    def plot_hyperparameter_impact(self):
        plt.figure(figsize=(16, 6))
        
        accuracies = [r['final_val_accuracy'] for r in self.results]
        labels = [f"LR={r['learning_rate']}\n{r['activation']}\nH1={r['hidden1_size']}" for r in self.results]
        colors = ['red' if acc == max(accuracies) else 'steelblue' for acc in accuracies]
        
        bars = plt.bar(range(len(accuracies)), accuracies, color=colors, alpha=0.7, edgecolor='black')
        plt.axhline(y=self.baseline_accuracy, color='green', linestyle='--', linewidth=2, label=f'Baseline ({self.baseline_accuracy:.2f}%)')
        plt.xlabel('Hyperparameter Combinations', fontsize=12)
        plt.ylabel('Validation Accuracy (%)', fontsize=12)
        plt.title('Hyperparameter Impact on Model Performance', fontsize=14, fontweight='bold')
        plt.xticks(range(len(labels)), labels, rotation=45, fontsize=9)
        plt.ylim([min(accuracies) - 2, 100])
        plt.legend(fontsize=10)
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_loss_curves(self):
        best_train_losses = self.best_result['train_losses']
        best_val_losses = self.best_result['val_losses']
        
        plt.figure(figsize=(14, 5))
        
        plt.subplot(1, 2, 1)
        epochs = range(1, len(self.baseline_train_losses) + 1)
        plt.plot(epochs, self.baseline_train_losses, label='Baseline Train', marker='o', linewidth=2)
        plt.plot(epochs, self.baseline_val_losses, label='Baseline Val', marker='s', linewidth=2)
        plt.plot(epochs, best_train_losses, label='Best Model Train', marker='o', linestyle='--', linewidth=2)
        plt.plot(epochs, best_val_losses, label='Best Model Val', marker='s', linestyle='--', linewidth=2)
        plt.xlabel('Epoch', fontsize=11)
        plt.ylabel('Loss', fontsize=11)
        plt.title('Training and Validation Loss Curves\n(Baseline vs Best Model)', fontsize=12, fontweight='bold')
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.plot(epochs, self.baseline_val_losses, label='Baseline Val Loss', marker='s', linewidth=2)
        plt.plot(epochs, best_val_losses, label='Best Model Val Loss', marker='s', linestyle='--', linewidth=2)
        plt.xlabel('Epoch', fontsize=11)
        plt.ylabel('Validation Loss', fontsize=11)
        plt.title('Validation Loss Comparison', fontsize=12, fontweight='bold')
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

    def plot_accuracy_curves(self):
        best_train_accuracies = self.best_result['train_accuracies']
        best_val_accuracies = self.best_result['val_accuracies']

        plt.figure(figsize=(14, 5))

        plt.subplot(1, 2, 1)
        epochs = range(1, len(self.baseline_train_losses) + 1)
        plt.plot(epochs, self.baseline_train_accuracies, label='Baseline Train Accuracy', marker='o', linewidth=2)
        plt.plot(epochs, self.baseline_val_accuracies, label='Baseline Val Accuracy', marker='s', linewidth=2)
        plt.plot(epochs, best_train_accuracies, label='Best Model Train Accuracy', marker='o', linestyle='--', linewidth=2)
        plt.plot(epochs, best_val_accuracies, label='Best Model Val Accuracy', marker='s', linestyle='--', linewidth=2)
        plt.xlabel('Epoch', fontsize=11)
        plt.ylabel('Accuracy(%)', fontsize=11)
        plt.title('Training and Validation Accuracy Curves\n(Baseline vs Best Model)', fontsize=12, fontweight='bold')
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.plot(epochs, self.baseline_train_accuracies, label='Baseline Val Accuracy', marker='s', linewidth=2)
        plt.plot(epochs, best_val_accuracies, label='Best Model Val Accuracy', marker='s', linestyle='--', linewidth=2)
        plt.xlabel('Epoch', fontsize=11)
        plt.ylabel('Validation Accuracy', fontsize=11)
        plt.title('Validation Accuracy Comparison', fontsize=12, fontweight='bold')
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        

    def plot_confusion_matrix(self, predictions, labels):
        cm = confusion_matrix(labels, predictions)
        
        plt.figure(figsize=(12, 10))
        plt.imshow(cm, interpolation='nearest', cmap='Blues')
        plt.title('Confusion Matrix - Best Model', fontsize=14, fontweight='bold')
        plt.colorbar()
        
        tick_marks = np.arange(len(self.class_names))
        plt.xticks(tick_marks, self.class_names, rotation=45, ha='right')
        plt.yticks(tick_marks, self.class_names)
        
        thresh = cm.max() / 2.
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, format(cm[i, j], 'd'),
                        ha="center", va="center",
                        color="white" if cm[i, j] > thresh else "black",
                        fontsize=9)
        
        plt.ylabel('True Label', fontsize=11)
        plt.xlabel('Predicted Label', fontsize=11)
        plt.tight_layout()
        plt.show()
        return cm

    def analyze_best_vs_baseline(self):
        print("\n" + "="*70)
        print("BEST vs. BASELINE")
        print("="*70)
        print(f"Best Hyperparameters:")
        print(f"  • Learning Rate: {self.best_result['learning_rate']}")
        print(f"  • Activation Function: {self.best_result['activation']}")
        print(f"  • Hidden Layer 1 Size: {self.best_result['hidden1_size']}")
        print(f"\nBaseline Accuracy: {self.baseline_accuracy:.2f}%")
        print(f"Best Model Accuracy: {self.best_result['final_val_accuracy']:.2f}%")
        improvement = self.best_result['final_val_accuracy'] - self.baseline_accuracy
        print(f"Improvement: {improvement:.2f}% ({improvement/self.baseline_accuracy*100:.1f}% relative improvement)")
        print("="*70)

    def analyze_hyperparameter_influence(self):
        print("\n" + "="*70)
        print("HYPERPARAMETER INFLUENCE")
        print("="*70)
        
        lr_impact = {}
        for lr in sorted(set(r['learning_rate'] for r in self.results)):
            lr_results = [r['final_val_accuracy'] for r in self.results if r['learning_rate'] == lr]
            lr_impact[lr] = (np.mean(lr_results), np.std(lr_results))
        
        print(f"\nLearning Rate Impact (avg ± std):")
        for lr, (mean, std) in lr_impact.items():
            print(f"  LR={lr}: {mean:.2f}% ± {std:.2f}%")
        
        best_lr = max(lr_impact.items(), key=lambda x: x[1][0])[0]
        print(f"  Best learning rate: {best_lr}")
        
        act_impact = {}
        for act in sorted(set(r['activation'] for r in self.results)):
            act_results = [r['final_val_accuracy'] for r in self.results if r['activation'] == act]
            act_impact[act] = (np.mean(act_results), np.std(act_results))
        
        print(f"\nActivation Function Impact (avg ± std):")
        for act, (mean, std) in act_impact.items():
            print(f"  {act}: {mean:.2f}% ± {std:.2f}%")
        
        best_act = max(act_impact.items(), key=lambda x: x[1][0])[0]
        print(f"  Best activation: {best_act}")
        
        h1_impact = {}
        for h1 in sorted(set(r['hidden1_size'] for r in self.results)):
            h1_results = [r['final_val_accuracy'] for r in self.results if r['hidden1_size'] == h1]
            h1_impact[h1] = (np.mean(h1_results), np.std(h1_results))
        
        print(f"\nHidden Layer 1 Size Impact (avg ± std):")
        for h1, (mean, std) in h1_impact.items():
            print(f"  {h1} neurons: {mean:.2f}% ± {std:.2f}%")
        
        best_h1 = max(h1_impact.items(), key=lambda x: x[1][0])[0]
        print(f"  Best hidden1 size: {best_h1}")
        print("="*70)

    def analyze_overfitting(self):
        print("\n" + "="*70)
        print("OVERFITTING ANALYSIS")
        print("="*70)
        
        best_train_losses = self.best_result['train_losses']
        best_val_losses = self.best_result['val_losses']
        
        print(f"\nBaseline Model:")
        print(f"  Final Train Loss: {self.baseline_train_losses[-1]:.4f}")
        print(f"  Final Val Loss: {self.baseline_val_losses[-1]:.4f}")
        
        print(f"\nBest Model:")
        print(f"  Final Train Loss: {best_train_losses[-1]:.4f}")
        print(f"  Final Val Loss: {best_val_losses[-1]:.4f}")
        
        gap_baseline = self.baseline_val_losses[-1] - self.baseline_train_losses[-1]
        gap_best = best_val_losses[-1] - best_train_losses[-1]
        
        print(f"\nTrain-Val Loss Gap (indicator of overfitting):")
        print(f"  Baseline gap: {gap_baseline:.4f}")
        print(f"  Best model gap: {gap_best:.4f}")
        
        if gap_best < gap_baseline:
            improvement = ((gap_baseline - gap_best) / gap_baseline) * 100
            print(f"  Hyperparameter tuning REDUCED overfitting by {improvement:.1f}%")
        elif gap_best > gap_baseline:
            print(f"  Hyperparameter tuning INCREASED overfitting")
        else:
            print(f"  Overfitting gap remains similar")
        print("="*70)

    def analyze_errors(self, confusion_matrix):
        print("\n" + "="*70)
        print("ERROR ANALYSIS (Confusion Matrix)")
        print("="*70)
        confused_pairs = []
        for i in range(len(confusion_matrix)):
            for j in range(len(confusion_matrix)):
                if i != j:
                    confused_pairs.append((confusion_matrix[i][j], self.class_names[i], self.class_names[j]))
        
        confused_pairs.sort(reverse=True)
        
        print(f"\nTop 2 Confused Class Pairs:")
        for idx, (count, true_class, pred_class) in enumerate(confused_pairs[:2], 1):
            print(f"  {idx}. {true_class} → {pred_class}: {count} misclassifications")
            
            confusion_reasons = {
                ('T-shirt/top', 'Shirt'): "Both have similar upper-body silhouettes and collar regions",
                ('Shirt', 'T-shirt/top'): "Both have similar upper-body silhouettes and collar regions",
                ('Coat', 'Pullover'): "Both are layering items with similar outlines and sleeves",
                ('Pullover', 'Coat'): "Both are layering items with similar outlines and sleeves",
                ('Sneaker', 'Ankle boot'): "Both cover the foot/ankle and have similar footwear shapes",
                ('Ankle boot', 'Sneaker'): "Both cover the foot/ankle and have similar footwear shapes",
                ('Trouser', 'Coat'): "Similar vertical orientation and length in low resolution",
                ('Sandal', 'Sneaker'): "Both are footwear items with similar overall profiles",
            }
            
            reason = confusion_reasons.get((true_class, pred_class), 
                                          "Similar visual features at low resolution (28x28)")
            print(f"     Reason: {reason}")
        
        print("\nGeneral Contributing Factors:")
        print("  • Low resolution (28x28) reduces distinguishing details")
        print("  • Grayscale images eliminate color differentiation")
        print("  • Similar clothing categories share structural similarities")
        print("="*70)

    def set_best_model(self, best_model):
        """Store reference to already-trained best model"""
        self.best_model = best_model

    def get_predictions(self, model, val_loader):
        """Get predictions from model on validation set"""
        model.eval()
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in val_loader:
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                all_predictions.extend(predicted.numpy())
                all_labels.extend(labels.numpy())
        
        return np.array(all_predictions), np.array(all_labels)

    def run_evaluation(self, train_loader, val_loader, test_loader):
        self.display_results_table()
        print("\nGenerating hyperparameter impact visualization...")
        self.plot_hyperparameter_impact()
        print("Generating loss curves...")
        self.plot_loss_curves()
        print("Generating accuracy curves...")
        self.plot_accuracy_curves()
        print("\nEvaluating best model on test set...")
        predictions, labels = self.get_predictions(self.best_model, test_loader)
        print("Generating confusion matrix...")
        cm = self.plot_confusion_matrix(predictions, labels)
        print("\n" + "="*70)
        print("Results Interpretation")
        print("="*70)
        self.analyze_best_vs_baseline()
        self.analyze_hyperparameter_influence()
        self.analyze_overfitting()
        self.analyze_errors(cm)

In [ ]:
if __name__ == "__main__":
    evaluator = Evaluate(results, baseline_accuracy, baseline_train_losses, baseline_val_losses)
    evaluator.set_best_model(best_result_model)  
    evaluator.run_evaluation(train_loader, val_loader, test_loader)